In [2]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

In [3]:
df= pd.read_csv("cleaned_dataset.csv")
dh= pd.read_csv("cleaned_testing.csv")

reader = Reader(rating_scale=(df['score'].min(), df['score'].max()))

data = Dataset.load_from_df(df[['customer_enc', 'song_enc', 'score']], reader)

In [4]:
trainset = data.build_full_trainset()  # use full train set
algo = SVD(n_factors=50, n_epochs=30, lr_all=0.005, reg_all=0.02, random_state=42)
algo.fit(trainset)

In [5]:
from surprise import Prediction
predictions = []
for _, row in df.iterrows():
    est = algo.predict(row['customer_enc'], row['song_enc']).est
    predictions.append(est)

In [6]:
mse = mean_squared_error(df['score'], predictions)
print("Surprise SVD MSE:", mse)

Surprise SVD MSE: 0.38928276004644435


In [7]:
from surprise import Prediction
predictions = []
for _, row in dh.iterrows():
    est = algo.predict(row['customer_enc'], row['song_enc']).est
    predictions.append(est)

In [8]:
sub_df = pd.DataFrame({
    'ID': range(len(dh)),  
    'score': predictions            
})

sub_df.to_csv('sub_new.csv', index=False)

In [9]:
from surprise.model_selection import GridSearchCV
param_grid = {
    'n_factors': [20, 50],
    'n_epochs': [20, 50],
    'lr_all': [0.005, 0.01],
    'reg_all': [0.05, 0.1]
}

gs = GridSearchCV(SVD, param_grid, measures=['rmse'], cv=3, n_jobs=-1)
gs.fit(data)

print("Best RMSE:", gs.best_score['rmse'])
print("Best hyperparameters:", gs.best_params['rmse'])

Best RMSE: 0.862840497939113
Best hyperparameters: {'n_factors': 50, 'n_epochs': 50, 'lr_all': 0.005, 'reg_all': 0.1}


In [10]:
best_params = gs.best_params['rmse']

algo = SVD(
    n_factors=best_params['n_factors'],
    n_epochs=best_params['n_epochs'],
    lr_all=best_params['lr_all'],
    reg_all=best_params['reg_all'],
    random_state=42
)

# Train on the full dataset
trainset = data.build_full_trainset()
algo.fit(trainset)

In [11]:
predictions = []
for _, row in dh.iterrows():
    pred = algo.predict(row['customer_enc'], row['song_enc']).est
    predictions.append(pred)


dh['predicted_rating'] = predictions

In [12]:
sub_df = pd.DataFrame({
    'ID': range(len(dh)),  
    'score': predictions            
})

sub_df.to_csv('sub_new2.csv', index=False)